In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from datetime import datetime



from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_score, recall_score, f1_score, make_scorer,
                             accuracy_score, classification_report, confusion_matrix)


from sklearn.preprocessing import StandardScaler, MinMaxScaler

# SMOTEENN

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE, ADASYN

from xgboost import XGBClassifier
from collections import Counter

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

import optuna
from sklearn.model_selection import cross_val_score



c:\Projects\Customer_churn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("..//data//raw//Customer-Churn.csv")

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.Churn.value_counts() / len(df) * 100

Churn
No     73.463013
Yes    26.536987
Name: count, dtype: float64

## Churn rate is 26.53 %

In [5]:
X = df.drop(columns= ['Churn', 'customerID'])

y = df.Churn

print("Shape of X : " , X.shape)
print("Shape of y : ", y.shape)

Shape of X :  (7043, 19)
Shape of y :  (7043,)


In [6]:
X = pd.get_dummies(X, drop_first= True)

y = df.Churn.map({'No' : 0, 'Yes' : 1})

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,stratify=y, random_state=42)

In [8]:
len(X_train)

5634

In [9]:
len(X_test)

1409

In [10]:
experiment_results = []


def log_experiment(
    model_name,
    approach,
    y_true,
    y_pred,
    cv_f1=None,
    best_params=None,
    notes=None
):
    """
    Store model evaluation results in a central list.
    """

    result = {
    "Model": model_name,
    "Approach": approach,
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Recall": recall_score(y_true, y_pred, zero_division=0),
    "F1": f1_score(y_true, y_pred, zero_division=0),
    "CV_F1_Mean": cv_f1,
    "Best_Params": str(best_params) if best_params else None,
    "Notes": notes
}

    experiment_results.append(result)

    return pd.DataFrame(experiment_results)

In [11]:
model_dt = DecisionTreeClassifier()
model_dt.fit(X_train, y_train)


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of sampl

In [12]:
y_pred_dt = model_dt.predict(X_test)

In [13]:
log_experiment(
    model_name="DecisionTree",
    approach="Baseline",
    y_true=y_test,
    y_pred=y_pred_dt,
    notes="Base line Decision Tree model"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.56875,0.486631,0.524496,None,None,Base line Decision Tree model


In [14]:
print(classification_report(y_true=y_test, y_pred=y_pred_dt))

              precision    recall  f1-score   support

           0       0.82      0.87      0.84      1035
           1       0.57      0.49      0.52       374

    accuracy                           0.77      1409
   macro avg       0.70      0.68      0.68      1409
weighted avg       0.76      0.77      0.76      1409



In [15]:
### Base model has an accuracy of 77 %

In [16]:
# Data Cleaning
teleco_data = df.copy()

In [17]:
teleco_data.TotalCharges = pd.to_numeric(teleco_data.TotalCharges, errors='coerce')

In [18]:
teleco_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [19]:
teleco_data.loc[teleco_data.TotalCharges.isnull() == True]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [20]:
teleco_data.dropna(how='any', inplace=True)

In [21]:
teleco_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [22]:
bins = [0,12,24,36,48, 60, 72]

labels = ['1-12', '13-24', '25-36', '37-48', '49-60', '61-72']

teleco_data['tenure_bin'] = pd.cut(teleco_data.tenure, bins=bins, labels=labels, include_lowest=True)

print(teleco_data[['tenure', 'tenure_bin']].head())
print(teleco_data.tenure_bin.value_counts().sort_index())

   tenure tenure_bin
0       1       1-12
1      34      25-36
2       2       1-12
3      45      37-48
4       2       1-12
tenure_bin
1-12     2175
13-24    1024
25-36     832
37-48     762
49-60     832
61-72    1407
Name: count, dtype: int64


In [23]:
teleco_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bin
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1-12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-36
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1-12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37-48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1-12


In [24]:
teleco_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   customerID        7032 non-null   object  
 1   gender            7032 non-null   object  
 2   SeniorCitizen     7032 non-null   int64   
 3   Partner           7032 non-null   object  
 4   Dependents        7032 non-null   object  
 5   tenure            7032 non-null   int64   
 6   PhoneService      7032 non-null   object  
 7   MultipleLines     7032 non-null   object  
 8   InternetService   7032 non-null   object  
 9   OnlineSecurity    7032 non-null   object  
 10  OnlineBackup      7032 non-null   object  
 11  DeviceProtection  7032 non-null   object  
 12  TechSupport       7032 non-null   object  
 13  StreamingTV       7032 non-null   object  
 14  StreamingMovies   7032 non-null   object  
 15  Contract          7032 non-null   object  
 16  PaperlessBilling  7032 non-nu

In [25]:
X = teleco_data.drop(columns= ['Churn', 'customerID', 'tenure'])

y = teleco_data.Churn

print("Shape of X : " , X.shape)
print("Shape of y : ", y.shape)

Shape of X :  (7032, 19)
Shape of y :  (7032,)


In [26]:
X = pd.get_dummies(X, drop_first= True)

y = teleco_data.Churn.map({'No' : 0, 'Yes' : 1})

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [28]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [29]:
model_dt2 = DecisionTreeClassifier()
model_dt2.fit(X_train, y_train)

y_pred_dt2 = model_dt2.predict(X_test)

In [30]:
log_experiment(
    model_name="DecisionTree",
    approach="StandardScaler",
    y_true=y_test,
    y_pred=y_pred_dt2,
    notes="Decision Tree after Standard Scaler"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler


In [31]:
print(classification_report(y_pred=y_pred_dt2, y_true=y_test))

              precision    recall  f1-score   support

           0       0.81      0.78      0.80      1033
           1       0.45      0.49      0.47       374

    accuracy                           0.71      1407
   macro avg       0.63      0.64      0.63      1407
weighted avg       0.71      0.71      0.71      1407



In [32]:
mms = MinMaxScaler()
X_train_mms = mms.fit_transform(X_train)
X_test_mms = mms.transform(X_test)

In [33]:
model_dt3 = DecisionTreeClassifier()
model_dt3.fit(X_train_mms, y_train)

y_pred_dt3 = model_dt3.predict(X_test_mms)

In [34]:
log_experiment(
    model_name="DecisionTree",
    approach="MinMaxScaler",
    y_true=y_test,
    y_pred=y_pred_dt3,
    notes="Decision Tree with Min Max"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max


In [35]:
print(classification_report(y_pred=y_pred_dt3, y_true=y_test))

              precision    recall  f1-score   support

           0       0.82      0.79      0.80      1033
           1       0.47      0.51      0.49       374

    accuracy                           0.71      1407
   macro avg       0.64      0.65      0.65      1407
weighted avg       0.72      0.71      0.72      1407



In [36]:
sm = SMOTEENN()
X_train_resampled, y_train_resampled = sm.fit_resample(X_train, y_train)

In [37]:
model_dt_smoteenn = DecisionTreeClassifier()
model_dt_smoteenn.fit(X_train_resampled, y_train_resampled)

y_pred_dt_smoteenn = model_dt_smoteenn.predict(X_test)

In [38]:
log_experiment(
    model_name="DecisionTree",
    approach="SMOTEENN",
    y_true=y_test,
    y_pred=y_pred_dt_smoteenn,
    notes="Decision Tree with smoteenn"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn


In [39]:
print(classification_report(y_pred=y_pred_dt_smoteenn, y_true=y_test))

              precision    recall  f1-score   support

           0       0.89      0.71      0.79      1033
           1       0.49      0.76      0.60       374

    accuracy                           0.73      1407
   macro avg       0.69      0.74      0.69      1407
weighted avg       0.79      0.73      0.74      1407



In [40]:
model_rf_smoteenn = RandomForestClassifier(n_estimators=500)
model_rf_smoteenn.fit(X_train_resampled, y_train_resampled)

y_pred_rf_smoteenn = model_rf_smoteenn.predict(X_test)

In [41]:
log_experiment(
    model_name="RandomForest",
    approach="SMOTEENN",
    y_true=y_test,
    y_pred=y_pred_rf_smoteenn,
    notes="Random Forest + SMOTEENN"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN


In [42]:
print(classification_report(y_pred=y_pred_rf_smoteenn, y_true=y_test))

              precision    recall  f1-score   support

           0       0.90      0.69      0.78      1033
           1       0.48      0.79      0.60       374

    accuracy                           0.72      1407
   macro avg       0.69      0.74      0.69      1407
weighted avg       0.79      0.72      0.73      1407



In [43]:
model_xgb = XGBClassifier(random_state = 42)
model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)



In [44]:
log_experiment(
    model_name="XGBoost",
    approach="Baseline",
    y_true=y_test,
    y_pred=y_pred_xgb,
    notes="Default XGBoost parameters"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters


In [45]:
print(classification_report(y_pred=y_pred_xgb, y_true=y_test))

              precision    recall  f1-score   support

           0       0.83      0.86      0.85      1033
           1       0.57      0.53      0.55       374

    accuracy                           0.77      1407
   macro avg       0.70      0.69      0.70      1407
weighted avg       0.76      0.77      0.77      1407



In [46]:
model_xgb_smoteenn = XGBClassifier(random_state = 42)
model_xgb_smoteenn.fit(X_train_resampled, y_train_resampled)

y_pred_xgb_smoteenn = model_xgb_smoteenn.predict(X_test)

print(classification_report(y_pred=y_pred_xgb_smoteenn, y_true=y_test))

              precision    recall  f1-score   support

           0       0.90      0.71      0.80      1033
           1       0.50      0.79      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.75      1407



In [47]:
log_experiment(
    model_name="XGBoost",
    approach="SMOTEENN",
    y_true=y_test,
    y_pred=y_pred_xgb_smoteenn,
    notes="XGBoost parameters with somte enn"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn


In [48]:
# checks class distribution using counter
print("Before smtoe: ", Counter(y_train))

smote = SMOTE(random_state=42)

X_train_smote , y_train_smote = smote.fit_resample(X_train, y_train)

print("After smote: ", Counter(y_train_smote))


model_xgb_smote = XGBClassifier(random_state = 42)
model_xgb_smote.fit(X_train_smote, y_train_smote)

y_pred_xgb_smote = model_xgb_smote.predict(X_test)

print(classification_report(y_pred=y_pred_xgb_smote, y_true=y_test))

Before smtoe:  Counter({0: 4130, 1: 1495})
After smote:  Counter({0: 4130, 1: 4130})
              precision    recall  f1-score   support

           0       0.85      0.84      0.84      1033
           1       0.57      0.57      0.57       374

    accuracy                           0.77      1407
   macro avg       0.71      0.71      0.71      1407
weighted avg       0.77      0.77      0.77      1407



In [49]:
log_experiment(
    model_name="XGBoost",
    approach="smote",
    y_true=y_test,
    y_pred=y_pred_xgb_smote,
    notes="XGB post smote"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote


In [50]:
# Check original distribution
print("Before ADASYN:", Counter(y_train))

# Apply ADASYN to training data only
adasyn = ADASYN(random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

# Check new distribution
print("After ADASYN:", Counter(y_train_adasyn))

# Train XGBoost on ADASYN-resampled data
model_xgb_adasyn = XGBClassifier(random_state=42)
model_xgb_adasyn.fit(X_train_adasyn, y_train_adasyn)

# Predict on the original test set
y_pred_xgb_adasyn = model_xgb_adasyn.predict(X_test)

# Evaluate
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_xgb_adasyn))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb_adasyn))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb_adasyn))

Before ADASYN: Counter({0: 4130, 1: 1495})
After ADASYN: Counter({0: 4130, 1: 4081})
Accuracy: 0.7633262260127932

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.83      0.84      1033
           1       0.55      0.57      0.56       374

    accuracy                           0.76      1407
   macro avg       0.70      0.70      0.70      1407
weighted avg       0.77      0.76      0.76      1407


Confusion Matrix:
 [[861 172]
 [161 213]]


In [51]:
log_experiment(
    model_name="XGBoost",
    approach="ADASYN",
    y_true=y_test,
    y_pred=y_pred_xgb_adasyn,
    notes="XGBoost parameters with ADASYN"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN


In [52]:
# Calculate scale_pos_weight (ratio of negative to positive class)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Train XGBoost with scale_pos_weight
model_xgb_weighted = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'  # avoids warnings
)

model_xgb_weighted.fit(X_train, y_train)
y_pred_weighted = model_xgb_weighted.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred_weighted))
print("\nClassification Report:\n", classification_report(y_test, y_pred_weighted))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_weighted))

scale_pos_weight: 2.76
Accuracy: 0.7476901208244492

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.77      0.82      1033
           1       0.52      0.69      0.59       374

    accuracy                           0.75      1407
   macro avg       0.70      0.73      0.70      1407
weighted avg       0.78      0.75      0.76      1407


Confusion Matrix:
 [[794 239]
 [116 258]]


In [53]:
log_experiment(
    model_name="XGBoost",
    approach="scale_pos_weight",
    y_true=y_test,
    y_pred=y_pred_weighted,
    notes="XGBoost parameters with scale pos weight"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [54]:
# Calculate the weight for the positive (minority) class
# Common approach: weight_positive = negative_count / positive_count
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
weight_positive = negative_count / positive_count if positive_count > 0 else 1
print(f"Negative class count: {negative_count}")
print(f"Positive class count: {positive_count}")
print(f"Weight for positive class: {weight_positive:.2f}")

# Create sample weights: higher weight for minority class
sample_weight = np.where(y_train == 1, weight_positive, 1.0)

# Train AdaBoost with sample weights
model_ada_weighted = AdaBoostClassifier(
    n_estimators=50,  # default, you can tune
    random_state=42
)

model_ada_weighted.fit(X_train, y_train, sample_weight=sample_weight)

# Predict and evaluate
y_pred_ada_weighted = model_ada_weighted.predict(X_test)

print("AdaBoost with sample weighting - Accuracy:", accuracy_score(y_test, y_pred_ada_weighted))
print("\nClassification Report:\n", classification_report(y_test, y_pred_ada_weighted))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_ada_weighted))

Negative class count: 4130
Positive class count: 1495
Weight for positive class: 2.76
AdaBoost with sample weighting - Accuracy: 0.7320540156361052

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.50      0.81      0.62       374

    accuracy                           0.73      1407
   macro avg       0.70      0.76      0.71      1407
weighted avg       0.80      0.73      0.75      1407


Confusion Matrix:
 [[726 307]
 [ 70 304]]


In [55]:
log_experiment(
    model_name="AdaBoost",
    approach="Sample Weighting",
    y_true=y_test,
    y_pred=y_pred_ada_weighted,
    notes="Default AdaBoost parameters"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [77]:
# Save the best model to a .pkl file
joblib.dump(model_ada_weighted, 'best_ada_boost_churn_model.pkl')

print("Best model successfully saved as 'best_optuna_churn_model.pkl'")

Best model successfully saved as 'best_optuna_churn_model.pkl'


In [56]:
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 400],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight]  # try both balanced and weighted
}

xgb = XGBClassifier(random_state=42, eval_metric='logloss')

# Use 'f1' or 'recall' scoring since churn is imbalanced
search = RandomizedSearchCV(
    xgb,
    param_grid,
    n_iter=50,
    cv=5,
    scoring='f1',  # or 'recall' if catching churn is priority
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best CV F1 score:", search.best_score_)

# Use best model
best_model = search.best_estimator_
y_pred_best = best_model.predict(X_test)
print("\nTest Classification Report:\n", classification_report(y_test, y_pred_best))

Best parameters: {'subsample': 0.8, 'scale_pos_weight': np.float64(2.762541806020067), 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.7}
Best CV F1 score: 0.6331700864298416

Test Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.82      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.76      0.70      1407
weighted avg       0.80      0.73      0.74      1407



In [57]:
log_experiment(
    model_name="XGBoost",
    approach="RandomizedSearchCV",
    y_true=y_test,
    y_pred=y_pred_best,
    notes="XGBoost parameters with randomized search cv"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [58]:
# Save this model

import joblib

# Save the best model to a .pkl file
joblib.dump(best_model, 'best_xgboost_churn_model.pkl')

print("Best model successfully saved as 'best_xgboost_churn_model.pkl'")

Best model successfully saved as 'best_xgboost_churn_model.pkl'


In [59]:
model_rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_estimators=300,
    max_depth=6
)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.90      0.69      0.78      1033
           1       0.48      0.80      0.60       374

    accuracy                           0.72      1407
   macro avg       0.69      0.74      0.69      1407
weighted avg       0.79      0.72      0.73      1407



In [60]:
log_experiment(
    model_name="RandomForest",
    approach="Class Weight",
    y_true=y_test,
    y_pred=y_pred_rf,
    notes="Default Random Forest parameters"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [61]:
from catboost import CatBoostClassifier

model_cat = CatBoostClassifier(
    auto_class_weights='Balanced',  # or 'SqrtBalanced'
    verbose=0,
    random_state=42
)
model_cat.fit(X_train, y_train)
y_pred_cat = model_cat.predict(X_test)
print(classification_report(y_test, y_pred_cat))

              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1033
           1       0.51      0.77      0.61       374

    accuracy                           0.74      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.79      0.74      0.76      1407



In [62]:
log_experiment(
    model_name="CatBoost",
    approach="Class Weight",
    y_true=y_test,
    y_pred=y_pred_cat,
    notes="Default CatBoost parameters"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [63]:
from lightgbm import LGBMClassifier

model_lgb = LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
model_lgb.fit(X_train, y_train)
y_pred_lgb = model_lgb.predict(X_test)
print(classification_report(y_test, y_pred_lgb))

[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000826 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 606
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
              precision    recall  f1-score   support

           0       0.89      0.75      0.81      1033
           1       0.52      0.75      0.61       374

    accuracy                           0.75      1407
   macro avg       0.71      0.75      0.71      1407
weighted avg       0.79      0.75      0.76      1407



In [64]:
log_experiment(
    model_name="LightGBM",
    approach="scale_pos_weight",
    y_true=y_test,
    y_pred=y_pred_lgb,
    notes="Default LightGBM parameters"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [65]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),  # L1 regularization
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2), # L2 regularization
        'scale_pos_weight': trial.suggest_categorical('scale_pos_weight', [1, scale_pos_weight]),
        'random_state': 42,
        'eval_metric': 'logloss'
    }

    model = XGBClassifier(**params)

    # Use F1 for minority class as scorer
    f1_scorer = make_scorer(f1_score, average='binary', pos_label=1)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring=f1_scorer, n_jobs=-1).mean()

    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)  # Increase trials for better results

print("Best parameters:", study.best_params)
print("Best CV F1 score:", study.best_value)

# Train final model with best params
best_model_optuna = XGBClassifier(**study.best_params)
best_model_optuna.fit(X_train, y_train)
y_pred_optuna = best_model_optuna.predict(X_test)
print(classification_report(y_test, y_pred_optuna))

[I 2026-09-28 13:42:21,651] A new study created in memory with name: no-name-159efdef-540d-4b9b-9f85-152fa344a4ad
[I 2026-09-28 13:42:24,066] Trial 0 finished with value: 0.5537041471903276 and parameters: {'max_depth': 9, 'learning_rate': 0.06599856987178462, 'n_estimators': 617, 'subsample': 0.8430644234009799, 'colsample_bytree': 0.639474130555922, 'min_child_weight': 6, 'gamma': 0.19509710286148502, 'reg_alpha': 0.6040254086911775, 'reg_lambda': 0.9289772813498343, 'scale_pos_weight': 1}. Best is trial 0 with value: 0.5537041471903276.
[I 2026-09-28 13:42:26,859] Trial 1 finished with value: 0.5853599050670807 and parameters: {'max_depth': 8, 'learning_rate': 0.2229663573229539, 'n_estimators': 999, 'subsample': 0.8967547065674091, 'colsample_bytree': 0.9378492229367507, 'min_child_weight': 2, 'gamma': 0.4982377010418283, 'reg_alpha': 0.7265601602232872, 'reg_lambda': 1.9727100565169533, 'scale_pos_weight': np.float64(2.762541806020067)}. Best is trial 1 with value: 0.5853599050670

Best parameters: {'max_depth': 3, 'learning_rate': 0.013277632179717964, 'n_estimators': 563, 'subsample': 0.7529755645106877, 'colsample_bytree': 0.9170866064868098, 'min_child_weight': 4, 'gamma': 0.18307137983777227, 'reg_alpha': 0.039949748009223705, 'reg_lambda': 1.1796586425064333, 'scale_pos_weight': np.float64(2.762541806020067)}
Best CV F1 score: 0.6391569786848607
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.80      0.72      0.74      1407



In [66]:
log_experiment(
    model_name="XGBoost",
    approach="Optuna",
    y_true=y_test,
    y_pred=y_pred_optuna,
    notes="XGBoost parameters with optuna"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [67]:
# Save the best model to a .pkl file
joblib.dump(best_model_optuna, 'best_optuna_churn_model.pkl')

print("Best model successfully saved as 'best_optuna_churn_model.pkl'")

Best model successfully saved as 'best_optuna_churn_model.pkl'


In [68]:
# Base estimator
ada_base = DecisionTreeClassifier(
    random_state=42
)

# Hyperparameter search space
ada_param_grid = {
    'n_estimators': [50, 100, 150, 200, 300, 400, 500],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2, 0.5, 1.0],

    # Hyperparameters of the base Decision Tree
    'estimator__max_depth': [1, 2, 3, 4, 5],
    'estimator__min_samples_split': [2, 5, 10, 20],
    'estimator__min_samples_leaf': [1, 2, 5, 10]
}

# AdaBoost model
ada = AdaBoostClassifier(
    estimator=ada_base,
    random_state=42
)

# Randomized Search
ada_search = RandomizedSearchCV(
    estimator=ada,
    param_distributions=ada_param_grid,
    n_iter=50,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

# Train
ada_search.fit(X_train, y_train)

# Results
print("Best parameters:")
print(ada_search.best_params_)

print("\nBest CV F1 score:")
print(ada_search.best_score_)

# Best model
best_ada_model = ada_search.best_estimator_

# Test prediction
y_pred_ada = best_ada_model.predict(X_test)

print("\nTest Classification Report:")
print(classification_report(y_test, y_pred_ada))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ada))

Best parameters:
{'n_estimators': 400, 'learning_rate': 0.5, 'estimator__min_samples_split': 10, 'estimator__min_samples_leaf': 2, 'estimator__max_depth': 3}

Best CV F1 score:
0.5953251727589182

Test Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.87      0.86      1033
           1       0.61      0.57      0.59       374

    accuracy                           0.79      1407
   macro avg       0.73      0.72      0.72      1407
weighted avg       0.78      0.79      0.78      1407


Confusion Matrix:
[[895 138]
 [162 212]]


In [69]:
log_experiment(
    model_name="AdaBoost",
    approach="RandomizedSearchCV",
    y_true=y_test,
    y_pred=y_pred_ada,
    notes="AdaBoost parameters with Randomized Search CV"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [70]:
def ada_objective(trial):

    # Tune the Decision Tree separately
    tree = DecisionTreeClassifier(
        max_depth=trial.suggest_int(
            'max_depth',
            1,
            5
        ),

        min_samples_split=trial.suggest_int(
            'min_samples_split',
            2,
            20
        ),

        min_samples_leaf=trial.suggest_int(
            'min_samples_leaf',
            1,
            10
        ),

        random_state=42
    )

    # Tune AdaBoost parameters
    model = AdaBoostClassifier(
        estimator=tree,

        n_estimators=trial.suggest_int(
            'n_estimators',
            50,
            500
        ),

        learning_rate=trial.suggest_float(
            'learning_rate',
            0.01,
            1.0,
            log=True
        ),

        random_state=42
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1',
        n_jobs=-1
    ).mean()

    return score


# Create Optuna study
ada_study = optuna.create_study(
    direction='maximize'
)

# Run optimization
ada_study.optimize(
    ada_objective,
    n_trials=100
)

print("Best parameters:")
print(ada_study.best_params)

print("\nBest CV F1 score:")
print(ada_study.best_value)

[I 2026-09-28 13:47:11,028] A new study created in memory with name: no-name-9128d5f1-0584-461a-8969-cbb6970e9c65
[I 2026-09-28 13:47:13,768] Trial 0 finished with value: 0.38695696593944656 and parameters: {'max_depth': 2, 'min_samples_split': 8, 'min_samples_leaf': 8, 'n_estimators': 208, 'learning_rate': 0.018102855139999166}. Best is trial 0 with value: 0.38695696593944656.
[I 2026-09-28 13:47:15,246] Trial 1 finished with value: 0.5700499064533501 and parameters: {'max_depth': 2, 'min_samples_split': 18, 'min_samples_leaf': 5, 'n_estimators': 107, 'learning_rate': 0.4357413720414119}. Best is trial 1 with value: 0.5700499064533501.
[I 2026-09-28 13:47:18,740] Trial 2 finished with value: 0.5860123532622011 and parameters: {'max_depth': 4, 'min_samples_split': 17, 'min_samples_leaf': 9, 'n_estimators': 175, 'learning_rate': 0.05447231644330217}. Best is trial 2 with value: 0.5860123532622011.
[I 2026-09-28 13:47:23,150] Trial 3 finished with value: 0.420547104614331 and parameters:

Best parameters:
{'max_depth': 3, 'min_samples_split': 19, 'min_samples_leaf': 3, 'n_estimators': 463, 'learning_rate': 0.29437156675733234}

Best CV F1 score:
0.6008638991395335


In [72]:
# Get best parameters from Optuna
best_params = ada_study.best_params

# Build the best Decision Tree
best_tree = DecisionTreeClassifier(
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    random_state=42
)

# Build the best AdaBoost model
best_ada_optuna = AdaBoostClassifier(
    estimator=best_tree,
    n_estimators=best_params["n_estimators"],
    learning_rate=best_params["learning_rate"],
    random_state=42
)

# Train
best_ada_optuna.fit(X_train, y_train)

# Test prediction
y_pred_ada_optuna = best_ada_optuna.predict(X_test)

print("AdaBoost Test Classification Report:")
print(
    classification_report(
        y_test,
        y_pred_ada_optuna
    )
)

print("\nAdaBoost Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_ada_optuna
    )
)

AdaBoost Test Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.63      0.52      0.57       374

    accuracy                           0.79      1407
   macro avg       0.73      0.71      0.72      1407
weighted avg       0.78      0.79      0.78      1407


AdaBoost Confusion Matrix:
[[916 117]
 [178 196]]


In [73]:
log_experiment(
    model_name="AdaBoost",
    approach="Optuna",
    y_true=y_test,
    y_pred=y_pred_ada_optuna,
    notes="AdaBoost parameters with Optuna"
)

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,DecisionTree,Baseline,0.765791,0.568750,0.486631,0.524496,None,None,Base line Decision Tree model
1,DecisionTree,StandardScaler,0.706468,0.452088,0.491979,0.471191,None,None,Decision Tree after Standard Scaler
2,DecisionTree,MinMaxScaler,0.714996,0.466993,0.510695,0.487867,None,None,Decision Tree with Min Max
3,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
4,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
5,XGBoost,Baseline,0.769723,0.572254,0.529412,0.550000,None,None,Default XGBoost parameters
6,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
7,XGBoost,smote,0.769723,0.565789,0.574866,0.570292,None,None,XGB post smote
8,XGBoost,ADASYN,0.763326,0.553247,0.569519,0.561265,None,None,XGBoost parameters with ADASYN
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


In [74]:
joblib.dump(
    best_ada_optuna,
    'best_adaboost_churn_model.pkl'
)

print(
    "Best AdaBoost model successfully saved "
    "as 'best_adaboost_churn_model.pkl'"
)

Best AdaBoost model successfully saved as 'best_adaboost_churn_model.pkl'


In [76]:
results_df = pd.DataFrame(experiment_results)

# Sort by F1 score
results_df = results_df.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

# Display
display(results_df)

# Create timestamp
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

file_name = f"churn_experiment_results_{timestamp}.xlsx"

# Export
with pd.ExcelWriter(
    file_name,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="Experiment Results",
        index=False
    )

print(f"Results exported to: {file_name}")

,Model,Approach,Accuracy,Precision,Recall,F1,CV_F1_Mean,Best_Params,Notes
0,AdaBoost,Sample Weighting,0.732054,0.497545,0.812834,0.617259,None,None,Default AdaBoost parameters
1,XGBoost,RandomizedSearchCV,0.728500,0.493528,0.815508,0.614919,None,None,XGBoost parameters with randomized search cv
2,LightGBM,scale_pos_weight,0.749112,0.519409,0.751337,0.614208,None,None,Default LightGBM parameters
3,CatBoost,Class Weight,0.743426,0.511586,0.767380,0.613904,None,None,Default CatBoost parameters
4,XGBoost,SMOTEENN,0.732765,0.498305,0.786096,0.609959,None,None,XGBoost parameters with somte enn
5,XGBoost,Optuna,0.724236,0.488636,0.804813,0.608081,None,None,XGBoost parameters with optuna
6,RandomForest,Class Weight,0.718550,0.482258,0.799465,0.601610,None,None,Default Random Forest parameters
7,RandomForest,SMOTEENN,0.718550,0.482026,0.788770,0.598377,None,None,Random Forest + SMOTEENN
8,DecisionTree,SMOTEENN,0.725657,0.489691,0.762032,0.596234,None,None,Decision Tree with smoteenn
9,XGBoost,scale_pos_weight,0.747690,0.519115,0.689840,0.592423,None,None,XGBoost parameters with scale pos weight


Results exported to: churn_experiment_results_2026-09-28_14-03-46.xlsx
